# SpecSynth AI — Thermal Conductivity Model Training
**Notebook:** `09_Thermal_Conductivity_Model_Training_v1.0.ipynb`  
**Release type:** Exploratory baseline  
**Target:** `target_thermal_conductivity_W_mK`

This notebook mirrors the supplied training script: it compares Random Forest and Extra Trees on the existing labelled modelling dataset, selects the model with the lowest validation MAE, refits on train + validation, evaluates once on the held-out test split, and saves the model and metrics.

**Important:** This is an exploratory baseline using the existing labelled dataset. The 75 Batches 01–15 research candidates are kept in research staging and are not included in this training run. Do not describe this model as production validated.


## 1. Environment and imports
Run this notebook from the `02_ML_Engine` folder, or adjust `DATASET_PATH` below.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Load the modelling dataset

Expected file:
`00_Data/processed/thermal_conductivity/thermal_conductivity_modeling_view_v1.6.0.csv`

This path follows the supplied training script. If your extracted folder has a different filename or location, change `DATASET_PATH`.


In [ ]:
DATASET_PATH = Path("00_Data/processed/thermal_conductivity/thermal_conductivity_modeling_view_v1.6.0.csv")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH.resolve()}\n"
        "Run the notebook from 02_ML_Engine or update DATASET_PATH."
    )

df = pd.read_csv(DATASET_PATH)
print("Shape:", df.shape)
display(df.head())


## 3. Define features and target

The feature list matches the supplied baseline training script. Target-derived fields such as thermal conductivity itself and derived thermal resistance are intentionally not included.


In [ ]:
NUM = [
    "thickness_mm",
    "density_kg_m3",
    "specific_heat_J_kgK",
    "vapour_resistance_factor_mu",
    "compressive_strength_kPa",
    "tensile_strength_perpendicular_kPa",
]

CAT = [
    "material_category_group",
    "material_family",
    "category",
    "manufacturer",
    "market",
    "lambda_level",
    "reaction_to_fire",
    "euroclass_main",
]

TARGET = "target_thermal_conductivity_W_mK"
SPLIT = "recommended_split"

required = NUM + CAT + [TARGET, SPLIT]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from dataset: {missing}")

df = df[df[TARGET].notna()].copy()
print("Labelled rows:", len(df))
print(df[SPLIT].value_counts(dropna=False))


## 4. Validate split integrity

The dataset is expected to have pre-assigned train, validation and test splits. This notebook uses those assignments and does not randomly reshuffle records.


In [ ]:
expected_splits = {"train", "validation", "test"}
actual_splits = set(df[SPLIT].dropna().unique())
if not expected_splits.issubset(actual_splits):
    raise ValueError(f"Expected splits {expected_splits}; found {actual_splits}")

splits = {s: df[df[SPLIT] == s].copy() for s in ["train", "validation", "test"]}
split_counts = {k: len(v) for k, v in splits.items()}
split_counts


## 5. Build preprocessing pipeline

- Numeric features: median imputation, with missingness indicators.
- Categorical features: most-frequent imputation and one-hot encoding.
- Unknown categories at inference time are ignored by the encoder.


In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True))
        ]), NUM),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CAT),
    ])

X_train = splits["train"][NUM + CAT]
y_train = splits["train"][TARGET]
X_val = splits["validation"][NUM + CAT]
y_val = splits["validation"][TARGET]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_validation:", X_val.shape, "y_validation:", y_val.shape)


## 6. Compare baseline models

Model selection uses validation MAE only. The test set remains held out until the final evaluation.


In [ ]:
candidate_estimators = {
    "RandomForest": RandomForestRegressor(
        n_estimators=500, min_samples_leaf=2, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=2, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

validation_scores = {}
fitted_candidates = {}

for name, estimator in candidate_estimators.items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    validation_scores[name] = {
        "MAE": mean_absolute_error(y_val, val_pred),
        "RMSE": mean_squared_error(y_val, val_pred) ** 0.5,
        "R2": r2_score(y_val, val_pred),
    }
    fitted_candidates[name] = pipe

validation_results = pd.DataFrame(validation_scores).T.sort_values("MAE")
display(validation_results)
selected_name = validation_results.index[0]
print("Selected model:", selected_name)


## 7. Refit selected model on train + validation

After choosing the model family using validation MAE, refit it on the combined train and validation records.


In [ ]:
selected_estimator_class = candidate_estimators[selected_name].__class__
final_model = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", selected_estimator_class(
        n_estimators=500, min_samples_leaf=2, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])

train_validation = pd.concat([splits["train"], splits["validation"]], axis=0)
final_model.fit(train_validation[NUM + CAT], train_validation[TARGET])
print("Final fit rows:", len(train_validation))


## 8. Evaluate on the held-out test set

These metrics are exploratory. Interpret them alongside the number of independent product groups, data provenance and potential dataset bias.


In [ ]:
X_test = splits["test"][NUM + CAT]
y_test = splits["test"][TARGET]
test_pred = final_model.predict(X_test)

test_metrics = {
    "MAE": mean_absolute_error(y_test, test_pred),
    "RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
    "R2": r2_score(y_test, test_pred),
}
print(test_metrics)

test_predictions = splits["test"].copy()
test_predictions["prediction_thermal_conductivity_W_mK"] = test_pred
test_predictions["absolute_error_W_mK"] = np.abs(
    test_predictions[TARGET] - test_predictions["prediction_thermal_conductivity_W_mK"]
)
display(test_predictions[[TARGET, "prediction_thermal_conductivity_W_mK", "absolute_error_W_mK"]].head(10))


## 9. Save model, metrics and predictions

Outputs are written to the existing ML engine structure. This notebook does not modify the SQLite master database.


In [ ]:
OUTPUT_ROOT = Path(".")
MODEL_DIR = OUTPUT_ROOT / "03_Models" / "thermal_prediction"
REPORT_DIR = OUTPUT_ROOT / "04_Evaluation" / "reports"
PRED_DIR = OUTPUT_ROOT / "04_Evaluation" / "predictions"

for folder in [MODEL_DIR, REPORT_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "thermal_conductivity_random_forest_exploratory_v1.0.joblib"
metrics_path = REPORT_DIR / "training_metrics_notebook_v1.0.json"
comparison_path = REPORT_DIR / "validation_model_comparison_notebook_v1.0.csv"
predictions_path = PRED_DIR / "test_predictions_notebook_v1.0.csv"

joblib.dump(final_model, model_path)

report = {
    "selected_model": selected_name,
    "validation": validation_scores,
    "test": test_metrics,
    "split_counts": split_counts,
    "target": TARGET,
    "feature_columns": NUM + CAT,
    "note": "Exploratory baseline. Not production validation.",
}
metrics_path.write_text(json.dumps(report, indent=2))
validation_results.to_csv(comparison_path, index=True)
test_predictions.to_csv(predictions_path, index=False)

print("Saved:")
print(model_path)
print(metrics_path)
print(comparison_path)
print(predictions_path)


## 10. Interpretation and next actions

Before production use:
1. Review errors by material family and manufacturer.
2. Check whether product-group splitting adequately tests generalisation to unseen products.
3. Verify source quality and measurement comparability.
4. Compare against additional models only after confirming the modelling dataset and split design.
5. Validate the inference contract before connecting this model to Revit.

**Do not use the 75 research-staging candidates as training data unless they are separately assessed as eligible and included in a new, versioned modelling export.**
